In [ ]:
import json

with open(
    "../data/candidate_profile.json",
    "r",
    encoding="utf-8"
) as f:
    candidate = json.load(f)


with open(
    "../data/interview_blueprint.json",
    "r",
    encoding="utf-8"
) as f:
    blueprint = json.load(f)

In [ ]:
print(candidate["name"])
print(blueprint["target_role"])

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

In [ ]:
class InterviewQuestion(BaseModel):
    question: str
    topic: str
    category: str
    difficulty: str

    question_type: str

    resume_reference: Optional[str] = None

    expected_concepts: list[str] = Field(
        default_factory=list
    )

    follow_up_possible: bool = True

In [ ]:
QUESTION_TYPES = [
    "conceptual",
    "technical_project",
    "problem_solving",
    "scenario",
    "behavioral",
    "follow_up"
]

In [ ]:
QUESTION_GENERATION_PROMPT = """
You are an expert technical interviewer.

Generate ONE interview question for the candidate.

The question must be based on:
- target role
- candidate profile
- interview blueprint
- requested topic
- requested difficulty
- requested question type

Rules:

1. Make the question specific to the candidate whenever useful.
2. Use resume evidence when generating project-related questions.
3. Never invent candidate experience.
4. Do not repeat previously asked questions.
5. Match the requested difficulty.
6. Ask only ONE primary question.
7. Avoid trivia unless the topic requires factual knowledge.
8. Prefer questions that reveal reasoning and understanding.
9. Do not include the answer.
10. Return ONLY valid JSON matching the InterviewQuestion schema.

Difficulty guidelines:

easy:
- Basic definitions
- Fundamental concepts
- Simple explanations

medium:
- Explain concepts in context
- Compare approaches
- Explain why a technique is appropriate
- Discuss basic trade-offs

hard:
- Design decisions
- Complex trade-offs
- Failure cases
- Scalability
- Mathematical reasoning
- Production scenarios

The requested difficulty is a hard constraint.
Do not generate a question substantially easier or harder than requested.
"""

In [ ]:
# target_topic = "SceneSense AI"
# difficulty = "easy"
# question_type = "technical_project"

In [ ]:
# question_input = {
#     "target_role": blueprint["target_role"],
#     "candidate_profile": candidate,
#     "priority_topics": blueprint["priority_topics"],
#     "topic": target_topic,
#     "difficulty": difficulty,
#     "question_type": question_type,
#     "previous_questions" : [
#     "What is overfitting?",
#     "How do you detect overfitting?"
# ]
# }

In [ ]:
# question_input_text = json.dumps(
#     question_input,
#     indent=2,
#     ensure_ascii=False
# )

In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv("../.env")

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)
print("API key loaded:", client.api_key is not None)

In [ ]:
from pydantic import BaseModel, Field


class GeneratedQuestion(BaseModel):
    question: str
    topic: str
    difficulty: str
    question_type: str
    expected_concepts: list[str] = Field(
        default_factory=list
    )

In [ ]:
schema = InterviewQuestion.model_json_schema()

In [ ]:
def generate_question(
    topic: str,
    difficulty: str,
    question_type: str,
    previous_questions: list[str] | None = None
):
    
    if previous_questions is None:
        previous_questions = []

    previous_text = "\n".join(
        f"- {q}"
        for q in previous_questions[-10:]
    )

    prompt = f"""
You are an expert technical interviewer.

Generate ONE interview question.

Target topic:
{topic}

Difficulty:
{difficulty}

Question type:
{question_type}

Previously asked questions:
{previous_text if previous_text else "None"}

Rules:

1. The question must test the specified topic.
2. Match the requested difficulty.
3. Do not repeat or closely rephrase a previous question.
4. The question should be appropriate for a technical interview.
5. Return the expected concepts that a strong answer should contain.
6. Return ONLY valid JSON.

Return:

{{
    "question": "...",
    "topic": "...",
    "difficulty": "...",
    "question_type": "...",
    "expected_concepts": [
        "...",
        "..."
    ]
}}
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert technical interviewer. "
                    "Return only valid JSON."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.7
    )

    response_text = (
        response.choices[0]
        .message
        .content
    )

    result = json.loads(
        response_text
    )

    return GeneratedQuestion.model_validate(
        result
    )

In [ ]:
def is_duplicate_question(
    new_question: str,
    previous_questions: list[str]
) -> bool:

    return new_question.strip().lower() in {
        q.strip().lower()
        for q in previous_questions
    }

In [ ]:
previous_questions = [
    "What is overfitting?"
]

print(
    is_duplicate_question(
        "What is overfitting?",
        previous_questions
    )
)

In [ ]:
print(
    is_duplicate_question(
        "How can you detect overfitting?",
        previous_questions
    )
)

## question generation

In [ ]:
question = generate_question(
    topic="Machine Learning",
    difficulty="medium",
    question_type="technical",
    previous_questions=[]
)

In [ ]:
print(question)

In [ ]:
previous_questions = [
    "What is overfitting?",
    "What is underfitting?",
    "How does regularization prevent overfitting?"
]

In [ ]:
question = generate_question(
    topic="Machine Learning",
    difficulty="medium",
    question_type="conceptual",
    previous_questions=previous_questions
)

print(question.question)

In [ ]:
easy_question = generate_question(
    topic="SQL",
    difficulty="easy",
    question_type="conceptual",
    previous_questions=[]
)

print(easy_question.question)

easy_question = generate_question(
    topic="SQL",
    difficulty="easy",
    question_type="conceptual",
    previous_questions=[]
)

print(easy_question.question)

hard_question = generate_question(
    topic="SQL",
    difficulty="hard",
    question_type="scenario",
    previous_questions=[]
)

print(hard_question.question)